# Smart Intersections using LLM as controller
STEPS:
1. Generate scenarios using SUMO program integrated in python
2. Preprocess the data from the scenarios in a dataset for the LLM
3. Train the LLM using the preprocessed dataset
4. Integrate the LLM as controller in Veins (as a HTTP local service and the called from a C++ module in Veins)
5. Visualize with SUMO-GUI from Veins integrated in python

In [5]:
# Imports
from pathlib import Path
import csv
import pandas as pd
import sys, subprocess, os
import importlib

from sumo_env import ensure_tools_in_path, find_sumo
from netgen import generate_grid_network
from routes import generate_random_routes
from config import write_sumocfg
from runner import run_sumo
from huggingface_hub import snapshot_download
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer


ModuleNotFoundError: No module named 'trl'

In [6]:
print(os.environ.get("SUMO_HOME"))


C:\Program Files (x86)\Eclipse\Sumo



STEP 1.1: Integrate SUMO with python

In this step, the project ensures that the SUMO framework is correctly integrated with the Python environment. The function performs this validation and, if necessary, adjusts the environment configuration to make SUMO importable within Python scripts.

In [ ]:

import os, sys
SUMO_HOME = r"C:\Program Files (x86)\Eclipse\Sumo" 
os.environ["SUMO_HOME"] = SUMO_HOME              
sys.path.append(SUMO_HOME + r"\tools")          

ensure_tools_in_path()
print("SUMO found at:", find_sumo("sumo"))

SUMO found at: C:\Program Files (x86)\Eclipse\Sumo\bin\sumo.EXE


STEP 1.2: Defining the Variables and Necessary Paths

The simulation parameters and file paths are defined to configure how the traffic scenario will be created and executed. The variable OUTDIR specifies where all generated files (networks, routes, configurations) will be saved. KIND determines whether the simulation uses a synthetic grid network or real OpenStreetMap data. Grid-related parameters such as GRID_SIZE, EDGE_LENGTH, SPEED, and LANES describe the network’s structure and traffic conditions, while OSM_FILE is used only when importing real map data. Additional variables like FLOWS, END_TIME, and SEED control the number of vehicles, simulation duration, and reproducibility. Finally, STEP_LENGTH and GUI define how the simulation is executed.

In [ ]:
# Parameters
OUTDIR = Path("notebooks/scenarios/demo_grid")

# Scenario kind: "grid" or "osm"
KIND = "grid"  # or "osm"

# Grid parameters
GRID_SIZE = 4
EDGE_LENGTH = 180.0
SPEED = 13.9      # m/s (~50 km/h)
LANES = 2
TURN_LANES = False

# OSM file (used only if KIND == "osm")
OSM_FILE = Path("data/osm/city.osm.pbf")  

# Traffic parameters
FLOWS = 900
END_TIME = 1800
SEED = 42
MIN_DISTANCE = 150.0

# Simulation parameters
STEP_LENGTH = 1.0
GUI = False  # set True to run sumo-gui


STEP 1.3: Generate the Network, Traffic Routes, and Configuration File

In this step, the code constructs the complete simulation scenario by generating the road network, traffic routes, and the corresponding SUMO configuration file. First, the output directory (OUTDIR) is created to store all generated files. If the scenario type is set to "grid", the generate_grid_network() function builds a structured network with intersections and roads defined by the previously set parameters (grid size, edge length, speed, and number of lanes). Next, generate_random_routes() creates vehicle trips and routes based on traffic flow characteristics such as the total number of vehicles, simulation time, and random seed for reproducibility. Finally, write_sumocfg() produces a SUMO configuration file that links the generated network and route data, along with simulation parameters like time step length.

In [9]:
OUTDIR.mkdir(parents=True, exist_ok=True)

# 1) Network
if KIND == "grid":
    net = generate_grid_network(
        OUTDIR,
        size=GRID_SIZE,
        length=EDGE_LENGTH,
        speed=SPEED,
        lanes=LANES,
        turn_lanes=TURN_LANES,
    )
    
# 2) Routes
trips, rou = generate_random_routes(
    OUTDIR, net,
    flows=FLOWS,
    end_time=END_TIME,
    seed=SEED,
    min_distance=MIN_DISTANCE,
)

# 3) Config
cfg = write_sumocfg(
    OUTDIR,
    net,
    rou,
    step_length=STEP_LENGTH,
    additionals=None,
)

net, rou, cfg


[netgen] C:\Program Files (x86)\Eclipse\Sumo\bin\netgenerate.EXE --grid --grid.number=4 --grid.length=180 --default.speed=13.9 --default.lanenumber=2 --tls.guess --no-turnarounds --output-file notebooks\scenarios\demo_grid\net.net.xml
[routes] Generating random trips/routes:
  c:\ai\pytorch_rtx5050\.venv\Scripts\python.exe C:\Program Files (x86)\Eclipse\Sumo\tools\randomTrips.py --net-file notebooks\scenarios\demo_grid\net.net.xml --end 1800 --period 2.0 --seed 42 --min-distance 150.0 --validate --trip-attributes departLane="best" departSpeed="max" departPos="base" --route-file notebooks\scenarios\demo_grid\routes.rou.xml --output-trip-file notebooks\scenarios\demo_grid\trips.trips.xml
[config] Wrote notebooks\scenarios\demo_grid\scenario.sumocfg


(WindowsPath('notebooks/scenarios/demo_grid/net.net.xml'),
 WindowsPath('notebooks/scenarios/demo_grid/routes.rou.xml'),
 WindowsPath('notebooks/scenarios/demo_grid/scenario.sumocfg'))

STEP 1.4: Run the SUMO Simulation and Collect Results

The generated scenario is executed in the SUMO traffic simulator, and performance metrics are collected. The function run_sumo(OUTDIR, cfg, gui=GUI) launches the simulation using the configuration file created earlier. Depending on the GUI parameter, the simulation can run either in graphical mode (with visual traffic flow) or in command-line mode for faster processing. During execution, SUMO models the movement of vehicles according to the defined routes and network structure, while the script records various output metrics such as vehicle speeds, travel times, waiting times, or traffic density.

In [12]:
metrics = run_sumo(OUTDIR, cfg, gui=GUI)
metrics

[runner] Starting SUMO (CLI) with scenario.sumocfg
[runner] Simulation finished: departed=900, arrived=900


{'departed': 900, 'arrived': 900}

STEP 1.5: Save Simulation Results to a Dataset

In this final step, the results collected from the SUMO simulation are stored in a structured dataset for later analysis. A new or existing CSV file is used to record both the scenario parameters and the resulting performance metrics. The code first checks if the file already exists and, if not, creates it with a header row. It then compiles all relevant information — such as network configuration, traffic settings, and simulation outputs into a single record.

In [11]:
dataset_csv = OUTDIR.parent / "dataset.csv"
is_new = not dataset_csv.exists()

row = {
    "scenario_path": str(OUTDIR),
    "kind": KIND,
    "grid_size": GRID_SIZE if KIND == "grid" else "",
    "edge_length": EDGE_LENGTH if KIND == "grid" else "",
    "speed": SPEED if KIND == "grid" else "",
    "lanes": LANES if KIND == "grid" else "",
    "turn_lanes": TURN_LANES if KIND == "grid" else "",
    "osm_file": str(OSM_FILE) if KIND == "osm" else "",
    "flows": FLOWS,
    "end": END_TIME,
    "seed": SEED,
    "min_distance": MIN_DISTANCE,
    "step_length": STEP_LENGTH,
    "gui": GUI,
    "departed": metrics.get("departed", ""),
    "arrived": metrics.get("arrived", ""),
}

with dataset_csv.open("a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(row.keys()))
    if is_new:
        w.writeheader()
    w.writerow(row)

# Preview dataset
df = pd.read_csv(dataset_csv)
df.tail()


,scenario_path,kind,grid_size,edge_length,speed,lanes,turn_lanes,osm_file,flows,end,seed,min_distance,step_length,gui,departed,arrived
0,notebooks\scenarios\demo_grid,grid,4,180.0,13.9,2,False,NaN,900,1800,42,150.0,1.0,False,900,900
1,notebooks\scenarios\demo_grid,grid,6,150.0,16.7,2,False,NaN,900,1800,42,150.0,1.0,False,900,900
2,notebooks\scenarios\demo_grid,grid,4,180.0,13.9,2,False,NaN,900,1800,42,150.0,1.0,False,900,900



Visualize the Simulation in SUMO-GUI


In [15]:
from sumo_env import find_sumo
import subprocess
sumo_gui = find_sumo("sumo-gui")
subprocess.Popen([sumo_gui, "-c", r"C:\Users\Denisa\Desktop\ACCS\Smart-Intersections-with-LLM\notebooks\scenarios\demo_grid\scenario.sumocfg"])


<Popen: returncode: None args: ['C:\\Program Files (x86)\\Eclipse\\Sumo\\bin...>

STEP 2.1: Running Multiple Simulations and Logging Vehicle Data

The project begins automated data collection by running multiple SUMO traffic simulations and recording detailed vehicle states at each simulation step. Then iterates through all scenario configuration files. For each scenario, it calls log_rollout_jsonl(), which runs the SUMO simulation (in non-GUI mode for efficiency) and logs the state of every vehicle, including position, speed, route, and other dynamic variables, at every simulation step. These records are stored in a JSON file, where each line corresponds to a single vehicle state update.

In [ ]:
import traci

def _cleanup_traci():
    try:
        for lab in list(traci.getConnectionLabels()):
            traci.switch(lab)
            traci.close(False)
    except Exception:
        pass

from pathlib import Path
from rollout_logger import log_rollout_jsonl

ROOT = Path("scenarios_batch")

for cfg in ROOT.rglob("scenario.sumocfg"):
    _cleanup_traci()
    scen = cfg.parent
    out_jsonl = scen / "vehicle_steps.jsonl"
    print(f"[run] {scen.name}")
    res = log_rollout_jsonl(
    outdir=scen,
    cfg_path=cfg,
    jsonl_path=Path("vehicle_steps.jsonl"), 
    gui=False,
    sample_every=1,
    max_steps=None,
    include_route=True
    )

    print(f"[ok] {res['steps_written']} steps → {out_jsonl}")


STEP 2.2: Reloading Local Modules for Active Development

In this step, the code sets up a flexible development workflow by allowing local modules to be reloaded without restarting the environment.

In [3]:
import importlib, sys
sys.path.append(".")
import rollout_logger
rollout_logger = importlib.reload(rollout_logger)


In [4]:
import sys, importlib
from pathlib import Path

sys.path.append(".")
import llm_utils as U
U = importlib.reload(U)

from rollout_logger import log_rollout_jsonl  # funcția de logging per-pas


STEP 2.3: Final Stage of the Data Generation Pipeline

This step completes the data preparation process for the SUMO-based traffic simulations. The code processes multiple generated scenarios, extracts all recorded vehicle trajectories, and converts them into a unified dataset suitable for training a LLM. First, U.ensure_rollouts() runs or reuses existing simulations to generate vehicle_steps.jsonl files containing detailed step-by-step vehicle states for each scenario. Next, U.concat_rollouts() merges all these individual logs into one large dataset (rollouts.jsonl). The combined data is then transformed by U.to_instruction_output() into an instruction–response format, where each entry represents a simulation state and its corresponding control action, and the dataset is split into training and validation sets.

In [ ]:
ROOT = Path("scenarios_batch")         
OUT_ROLL = Path("data/rollouts.jsonl")
TRAIN = Path("data/train.jsonl"); VAL = Path("data/val.jsonl")

# Generează <scen>/vehicle_steps.jsonl 
U.ensure_rollouts(ROOT, log_fn=log_rollout_jsonl, sample_every=1, max_steps=None, gui=False)

# Concatenează toate în dataset mare
U.concat_rollouts(ROOT, OUT_ROLL)

# Transformă în instruction/output + split
U.to_instruction_output(OUT_ROLL, TRAIN, VAL, val_ratio=0.1, seed=42)

# Preview rapid
U.preview_jsonl(TRAIN, 2)
U.preview_jsonl(VAL, 2)


STEP 3.1: Proceseaza setul de date sa fie sub forma de intructiuni ca sa fie potrivit pt input ul llm ului

In [ ]:
import json
from pathlib import Path

path = Path("data/train.jsonl")  

with path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print("RAW:", line.strip())
        obj = json.loads(line)
        print("instruction:", obj.get("instruction")[:120], "...")
        print("output     :", obj.get("output"))
        print("-" * 40)


RAW: {"instruction": "{\"step\": 1372, \"veh_id\": \"679\", \"edge_id\": \"A0A1\", \"lane_index\": 0, \"lane_pos\": 7.450144670142838, \"speed\": 4.879742856032029, \"accel\": 1.7504784131422642, \"waiting_time\": 0.0, \"leader_gap\": 32.15488857262527, \"tls_id\": \"A1\", \"tls_dist\": 6.0, \"tls_state\": \"G\", \"dest_edge\": \"E0F0\"}", "output": "{\"action_speed\": \"accelerate\", \"action_lane\": \"keep_lane\"}"}
instruction: {"step": 1372, "veh_id": "679", "edge_id": "A0A1", "lane_index": 0, "lane_pos": 7.450144670142838, "speed": 4.8797428560 ...
output     : {"action_speed": "accelerate", "action_lane": "keep_lane"}
----------------------------------------
RAW: {"instruction": "{\"step\": 1230, \"veh_id\": \"562\", \"edge_id\": \"D2D3\", \"lane_index\": 0, \"lane_pos\": 128.1989124257164, \"speed\": 0.0, \"accel\": 0.0, \"waiting_time\": 12.0, \"leader_gap\": null, \"tls_id\": \"D3\", \"tls_dist\": 8.0, \"tls_state\": \"r\", \"dest_edge\": \"D3E3\"}", "output": "{\"action_speed

STEP 3.2: Download la modelul utilizat.

In [6]:
import os
from huggingface_hub import snapshot_download

def download_model(repo_id):
    base_dir = os.path.join("outputs", "models")
    model_name = repo_id.split("/")[-1]
    local_dir = os.path.join(base_dir, model_name)
    os.makedirs(local_dir, exist_ok=True)
    snapshot_download(repo_id=repo_id, local_dir=local_dir)
    
    

MODEL_SELECTOR = {
    1 : lambda: download_model("Qwen/Qwen2.5-1.5B"),
    2 : lambda: download_model("Qwen/Qwen2.5-1.5B-Instruct")

}

model_id = 2    
MODEL_SELECTOR[model_id]()

print("Model downloaded succesfully!")

Fetching 10 files: 100%|██████████| 10/10 [00:00<?, ?it/s]

Model downloaded succesfully!


STEP 3.3: Incarcarea modelului pe GPU

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_DIR = r"C:\Users\Denisa\Desktop\ACCS\Smart-Intersections-with-LLM\outputs\models\Qwen2.5-1.5B-Instruct"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.to(device)

print("Model and tokenizer loaded.")


c:\ai\pytorch_rtx5050\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Model and tokenizer loaded.


STEP 3.4: Configurare LORA pentru antenare

In [2]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


STEP 3.5: formatarea setului de date

In [ ]:
from datasets import load_dataset


data_files = {
    "train": "data/train.jsonl",
    "validation": "data/val.jsonl",
}

raw_datasets = load_dataset("json", data_files=data_files)
print(raw_datasets)
print("Un exemplu brut:", raw_datasets["train"][0])

def format_example(example):
    instr = example["instruction"]
    out = example["output"]
    text = f"State:\n{instr}\n\nAction:\n{out}"
    return {"text": text}

column_names = raw_datasets["train"].column_names

lm_datasets = raw_datasets.map(
    format_example,
    remove_columns=column_names,
)

print(lm_datasets["train"][0])
print("Exemple train:", len(lm_datasets["train"]))
print("Exemple val:", len(lm_datasets["validation"]))


DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 97649
    })
    validation: Dataset({
        features: ['instruction', 'output'],
        num_rows: 10849
    })
})
Un exemplu brut: {'instruction': '{"step": 1372, "veh_id": "679", "edge_id": "A0A1", "lane_index": 0, "lane_pos": 7.450144670142838, "speed": 4.879742856032029, "accel": 1.7504784131422642, "waiting_time": 0.0, "leader_gap": 32.15488857262527, "tls_id": "A1", "tls_dist": 6.0, "tls_state": "G", "dest_edge": "E0F0"}', 'output': '{"action_speed": "accelerate", "action_lane": "keep_lane"}'}
{'text': 'State:\n{"step": 1372, "veh_id": "679", "edge_id": "A0A1", "lane_index": 0, "lane_pos": 7.450144670142838, "speed": 4.879742856032029, "accel": 1.7504784131422642, "waiting_time": 0.0, "leader_gap": 32.15488857262527, "tls_id": "A1", "tls_dist": 6.0, "tls_state": "G", "dest_edge": "E0F0"}\n\nAction:\n{"action_speed": "accelerate", "action_lane": "keep_lane"}'}
Exemple train: 97649


STEP 3.6: Maparea setului de date

In [4]:
MAX_LENGTH = 256

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = lm_datasets.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
)

print(tokenized_datasets["train"][0])


Map: 100%|██████████| 10849/10849 [00:00<00:00, 19562.48 examples/s]

{'input_ids': [1397, 510, 4913, 9520, 788, 220, 16, 18, 22, 17, 11, 330, 15200, 842, 788, 330, 21, 22, 24, 497, 330, 7186, 842, 788, 330, 32, 15, 32, 16, 497, 330, 37847, 3560, 788, 220, 15, 11, 330, 37847, 6479, 788, 220, 22, 13, 19, 20, 15, 16, 19, 19, 21, 22, 15, 16, 19, 17, 23, 18, 23, 11, 330, 20374, 788, 220, 19, 13, 23, 22, 24, 22, 19, 17, 23, 20, 21, 15, 18, 17, 15, 17, 24, 11, 330, 43888, 788, 220, 16, 13, 22, 20, 15, 19, 22, 23, 19, 16, 18, 16, 19, 17, 17, 21, 19, 17, 11, 330, 49534, 3009, 788, 220, 15, 13, 15, 11, 330, 37391, 51790, 788, 220, 18, 17, 13, 16, 20, 19, 23, 23, 23, 20, 22, 17, 21, 17, 20, 17, 22, 11, 330, 34488, 842, 788, 330, 32, 16, 497, 330, 34488, 16031, 788, 220, 21, 13, 15, 11, 330, 34488, 4387, 788, 330, 38, 497, 330, 4979, 17932, 788, 330, 36, 15, 37, 15, 63159, 2512, 510, 4913, 1311, 16944, 788, 330, 43888, 58668, 497, 330, 1311, 60302, 788, 330, 13096, 60302, 9207], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

STEP 3.7: Antrenarea efectiva a modelului

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="models/qwen2_1_5b_lora_finetuned_new",
    overwrite_output_dir=True,
    num_train_epochs=1,               
    per_device_train_batch_size=1,    
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=(device.type == "cuda"),
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer.train()


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
50,1.486500
100,0.816600
150,0.771200
200,0.752900
250,0.764200
300,0.754700
350,0.747500
400,0.732300
450,0.740800
500,0.754600


TrainOutput(global_step=12207, training_loss=0.7079228464231119, metrics={'train_runtime': 14310.9097, 'train_samples_per_second': 6.823, 'train_steps_per_second': 0.853, 'total_flos': 1.291829299238953e+17, 'train_loss': 0.7079228464231119, 'epoch': 1.0})